# Detour · Python for QEC, Part I

Notes: [Detour · Python for QEC](https://claude.ai/artifact/5xz3vPhVFd8FBsxYomWF8E), Notes and Exercises tabs. Part I covers the Python that NB1 uses.

**Play, read, build.**
1. *Play (about 20 min):* Kernel → Restart Kernel and Run All Cells. Every worked-example cell (W1.1 to W7.2) runs
   without any code of yours. The exercise cells at the bottom print "not written yet" until you write the drills.
2. *Read:* one notes section at a time, with its W cells open. Change numbers in the cells and rerun.
3. *Build:* answer the section's "predict" questions on paper, then write that set in `drills.py`:
   `pytest detours/python -q -k p1` (then `p2` ... `p7`), and rerun its X cell.

Nothing here calls the five NB1 TODOs in `msd/toric.py`; those stay yours.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib, itertools, math, time
from collections import Counter
from typing import Optional

# Walk up from the notebook's folder to the repo root (the folder that holds msd/).
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "msd").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from msd.toric import (Decoder, css_parameters, cycle_code, gf2_rank, hedge_index, hypergraph_product,
                       logical_rate_terms, plot_lattice, repetition_code, support_to_vector, syndrome_key,
                       syndrome_of, vector_to_support, vedge_index)
from detours.python import drills as dr
print("repo root:", ROOT)

## §1 Arrays as F₂ vectors

In [ ]:
# W1.1  The Hamming [7,4] parity-check matrix, built by broadcasting
cols = np.arange(1, 8)            # shape (7,): column j will hold the number j + 1
bits = np.arange(3)               # shape (3,): row i will hold bit i
H = ((cols[None, :] >> bits[:, None]) & 1).astype(np.uint8)   # (1, 7) and (3, 1) broadcast to (3, 7)
print(H)
print("shape", H.shape, " dtype", H.dtype, " n =", H.shape[1], " checks =", H.shape[0])
print("column 4:", H[:, 4], " = 5 in binary, bit 0 first")
print("row 0   :", H[0], " -> check 0 looks at positions", np.flatnonzero(H[0]))

In [ ]:
# W1.2  An error, its support, its weight and its syndrome
n = H.shape[1]
e = support_to_vector(n, [4])                  # flip bit 4
print("e        =", e, " support", vector_to_support(e), " weight", int(e.sum()))
s = syndrome_of(H, e)
print("syndrome =", s, " read as a number:", int(s @ (1 << bits)))   # 1*s0 + 2*s1 + 4*s2
print("support_to_vector(n, [4, 4, 6]) ->", vector_to_support(support_to_vector(n, [4, 4, 6])), " (repeats cancel)")

In [ ]:
# W1.3  Views and copies
A = H.copy()                       # an independent copy of H
row = A[0]                         # basic indexing (an integer or a slice): a VIEW into A
row[0] ^= 1                        # ... so this edits A itself
print("A[0] after editing the view:", A[0], "  shares memory:", np.shares_memory(row, A))
two = A[[0, 1]]                    # fancy indexing (a list of rows): a COPY
two[:] = 0
print("A[:2] after zeroing the copy:\n", A[:2], "  shares memory:", np.shares_memory(two, A))
A[[0, 2]] = A[[2, 0]]              # swap rows 0 and 2, as gf2_rank does
print("A after swapping rows 0 and 2:\n", A)
print("H untouched:", np.array_equal(H[0], [1, 0, 1, 0, 1, 0, 1]))

## §2 Mod-2 arithmetic and the dtype traps

In [ ]:
# W2.1  A generator matrix, the check G H^T = 0, and one encoding
G = np.zeros((4, 7), dtype=np.uint8)
for i, sup in enumerate([(0, 1, 2), (0, 3, 4), (1, 3, 5), (0, 1, 3, 6)]):
    G[i, list(sup)] = 1
print(G)
GHt = G.astype(np.int64) @ H.T.astype(np.int64) % 2
print("G H^T mod 2 =\n", GHt, "\nany nonzero entry?", GHt.any())
m = np.array([1, 0, 1, 1], dtype=np.uint8)
c = m.astype(np.int64) @ G % 2
print("message", m, "-> codeword", c, " syndrome", syndrome_of(H, c))

In [ ]:
# W2.2  The four bitwise operators, read over F2
c1, c2 = G[0], G[1]
print("c1      ", c1)
print("c2      ", c2)
print("c1 ^ c2 ", c1 ^ c2, " sum in F2;  syndrome", syndrome_of(H, c1 ^ c2), "-> a codeword")
print("c1 & c2 ", c1 & c2, " overlap;    c1 . c2 =", int((c1 & c2).sum() % 2), "(inner product mod 2)")
print("c1 | c2 ", c1 | c2, " union;      syndrome", syndrome_of(H, c1 | c2), "-> NOT a codeword")

In [ ]:
# W2.3  Three dtype traps
e = support_to_vector(7, (0, 2))           # bits 0 and 2 both sit under check 0
print("correct              :", syndrome_of(H, e))
print("bool @ bool          :", (H.astype(bool) @ e.astype(bool)).astype(np.uint8), " <- OR, not XOR")
print("uint8 @ then % 2     :", H @ e % 2, " (fine: wrap-around is by 256, an even number)")
ones = np.ones(300, dtype=np.uint8)
print("uint8 dot, 300 ones  :", ones @ ones, "   but ones.sum() =", ones.sum(), " <- counts break, parities don't")
print("uint8 [0] - 1        :", np.array([0], dtype=np.uint8) - 1, " <- silent wrap-around")

## §3 GF(2) linear algebra in code

In [ ]:
# W3.1  gf2_rank, traced: the loop from msd/toric.py with a print after each column
def rank_trace(M):
    A = (np.array(M, dtype=np.uint8) % 2).copy()
    rows, cols = A.shape
    r = 0                                          # rows finished so far = rank so far
    for c in range(cols):
        if r == rows:
            break
        pivots = np.flatnonzero(A[r:, c])          # unfinished rows with a 1 in column c
        if pivots.size == 0:
            print(f"column {c}: no 1 in rows {r}.., skip")
            continue
        p = r + int(pivots[0])
        if p != r:
            A[[r, p]] = A[[p, r]]                  # move the pivot row up to row r
        hit = np.flatnonzero(A[:, c])
        hit = hit[hit != r]
        A[hit] ^= A[r]                             # clear column c in every other row
        print(f"column {c}: pivot in row {r} (was row {p}), added it to rows {hit.tolist()}\n{A}")
        r += 1
    return r

M = np.vstack([H, H[0] ^ H[1]])                    # 4 rows, but row 3 = row 0 + row 1
print("M =\n", M)
print("rank =", rank_trace(M))

In [ ]:
# W3.2  k from ranks, and Kronecker products
print("Hamming: n = 7, rank H =", gf2_rank(H), "-> k = n - rank H =", 7 - gf2_rank(H))
B = np.array([[1, 1, 0]], dtype=np.uint8)
print("kron(B, I_2): each 1 of B becomes a copy of I_2, each 0 a zero block\n",
      np.kron(B, np.eye(2, dtype=np.uint8)))
R3 = repetition_code(3)
HX, HZ = hypergraph_product(R3, R3)
print("HGP(R3, R3): H_X", HX.shape, " H_Z", HZ.shape)
print("CSS condition H_X H_Z^T = 0:", not (HX.astype(np.int64) @ HZ.T.astype(np.int64) % 2).any())
print("(n, k) =", css_parameters(HX, HZ), "  k = 13 -", gf2_rank(HX), "-", gf2_rank(HZ))

In [ ]:
# W3.3  Is a word a sum of rows of G? Compare ranks.
c = np.array([1, 1, 0, 1]) @ G % 2            # a codeword
x = support_to_vector(7, [0])                  # a weight-1 word
print("rank G      =", gf2_rank(G))
print("rank [G; c] =", gf2_rank(np.vstack([G, c])), " -> c is in the row space")
print("rank [G; x] =", gf2_rank(np.vstack([G, x])), " -> x is not")
print("cheaper here, because row space of G = ker H:  H c =", syndrome_of(H, c), "  H x =", syndrome_of(H, x))

## §4 Syndromes as keys

In [ ]:
# W4.1  Arrays cannot be dictionary keys; bytes can
s = syndrome_of(H, support_to_vector(7, [4]))
try:
    table = {s: (4,)}
except TypeError as err:
    print("TypeError:", err)
print("tuple key :", tuple(s.tolist()))
print("bytes key :", syndrome_key(s), " from packbits", np.packbits(s), "= 0b10100000 (3 bits padded to a byte)")
print("padding   : key([1,0]) == key([1,0,0])?", syndrome_key(np.array([1, 0])) == syndrome_key(np.array([1, 0, 0])))
HX5, HZ5 = hypergraph_product(cycle_code(5), cycle_code(5))
s5 = syndrome_of(HZ5, support_to_vector(50, [0]))
print("on T_5 a syndrome has", s5.size, "bits -> a key of", len(syndrome_key(s5)), "bytes")

In [ ]:
# W4.2  Bits as Python integers: a syndrome is the XOR of its columns
weights = 1 << np.arange(3)                          # [1 2 4]: row i is bit i
col_int = [int(H[:, j] @ weights) for j in range(7)]
print("columns as integers:", col_int)
acc = 0
for j in (0, 2):
    acc ^= col_int[j]
print("syndrome of {0, 2} as an integer: 1 ^ 3 =", acc, "  as a vector:", syndrome_of(H, support_to_vector(7, (0, 2))))
big_endian = [int.from_bytes(np.packbits(H[:, j]).tobytes(), "big") for j in range(7)]
print("is_degenerate's encoding (packbits, big-endian):", big_endian)

In [ ]:
# W4.3  Keep the first, keep the last, group, count
pairs = [(syndrome_key(syndrome_of(H, support_to_vector(7, sup))), sup) for sup in [(4,), (0,), (0, 3)]]
first, last, groups = {}, {}, {}
for k, sup in pairs:                   # (4,) and (0, 3) have the same syndrome
    first.setdefault(k, sup)           # writes only if k is new
    last[k] = sup                      # always overwrites
    groups.setdefault(k, []).append(sup)   # NB1 cell 15 groups errors this way
print("setdefault keeps the first:", first)
print("assignment keeps the last :", last)
print("grouped                   :", groups)
print("counted                   :", Counter(k for k, _ in pairs))

## §5 Enumerating errors, and what it costs

In [ ]:
# W5.1  combinations: lexicographic order, and how many there are
print(list(itertools.combinations(range(5), 2)))
print("C(5, 2) =", math.comb(5, 2))
for name, nn in [("T_5", 50), ("T_7", 98), ("T_9", 162)]:
    for w_max in (3, 4):
        total = sum(math.comb(nn, w) for w in range(w_max + 1))
        print(f"{name}: n = {nn:3d}, all weights <= {w_max}: {total:>12,d} supports")

In [ ]:
# W5.2  Generators are lazy: nothing is built until you ask for it
gen = itertools.combinations(range(162), 4)          # instant; holds no list
print("first three:", list(itertools.islice(gen, 3)))
print("next one   :", next(gen))
N = math.comb(162, 4)
per = sys.getsizeof((0, 1, 2, 3)) + 8                # one 4-tuple + one list slot, in bytes
print(f"list(...) would hold {N:,} tuples: about {N * per / 1e9:.1f} GB")

In [ ]:
# W5.3  One Python call per error, or one matrix product for all of them
sups = list(itertools.combinations(range(50), 3))    # every weight-3 error on T_5
t0 = time.perf_counter()
S_loop = np.array([syndrome_of(HZ5, support_to_vector(50, sup)) for sup in sups])
t1 = time.perf_counter()
idx = np.array(sups)                                 # shape (19600, 3)
E = np.zeros((len(sups), 50), dtype=np.uint8)
E[np.arange(len(sups))[:, None], idx] = 1            # row i gets ones at columns idx[i]
S_batch = E.astype(np.int64) @ HZ5.T.astype(np.int64) % 2
t2 = time.perf_counter()
print(f"{len(sups):,} errors: loop {t1 - t0:.3f} s, batch {t2 - t1:.4f} s, "
      f"{(t1 - t0) / (t2 - t1):.0f}x faster, same answer: {np.array_equal(S_loop, S_batch)}")
print(f"memory: E holds {E.nbytes / 1e6:.1f} MB as uint8, {E.nbytes * 8 / 1e6:.1f} MB once cast to int64")

## §6 Decoders as functions

In [ ]:
# W6.1  A decoder maps a syndrome to a correction. A factory function builds one.
def make_repetition_decoder(n: int) -> Decoder:
    """Minimum-weight decoder for the length-n repetition code (n odd)."""
    def decode(s: np.ndarray) -> Optional[np.ndarray]:
        if not s.any():
            return np.zeros(n, dtype=np.uint8)       # uses n from the enclosing call
        # s[i] = e[i] + e[i+1]. Fix e[0] = 0 and add up: e0 explains s, and so does e0 ^ 1.
        e0 = np.concatenate([[0], np.cumsum(s) % 2]).astype(np.uint8)
        e1 = e0 ^ 1
        return e0 if e0.sum() <= e1.sum() else e1
    return decode

R5 = repetition_code(5)
decode5 = make_repetition_decoder(5)
print("decode5 remembers n =", decode5.__closure__[0].cell_contents)
for sup in [(1, 3), (0, 1, 3)]:
    e = support_to_vector(5, sup)
    e_hat = decode5(syndrome_of(R5, e))
    print(f"error {sup}: syndrome {syndrome_of(R5, e)}, correction {vector_to_support(e_hat)} ->",
          "exact" if np.array_equal(e_hat, e) else f"wrong, e + e_hat = {e ^ e_hat}")

In [ ]:
# W6.2  None means "I give up"; test it with `is None`
def cautious(s: np.ndarray) -> Optional[np.ndarray]:
    """Handles only the trivial syndrome."""
    return np.zeros(5, dtype=np.uint8) if not s.any() else None

for dec in (decode5, cautious):
    e_hat = dec(syndrome_of(R5, support_to_vector(5, [2])))
    print(f"{dec.__name__:8s} ->", "gave up" if e_hat is None else vector_to_support(e_hat))
try:
    if np.zeros(5):
        pass
except ValueError as err:
    print("`if array:` fails:", err)

keep = lambda sup: 0 in sup                          # a predicate, like NB1's keep=
print("weight-2 supports kept:", [sup for sup in itertools.combinations(range(5), 2) if keep(sup)])

In [ ]:
# W6.3  Late binding: lambdas made in a loop all see the loop variable's LAST value
decoders = [lambda s: n for n in (3, 5, 7)]
print("lambdas made in a loop  :", [dec(None) for dec in decoders])
decoders = [make_repetition_decoder(n) for n in (3, 5, 7)]
print("factory called in a loop:", [dec.__closure__[0].cell_contents for dec in decoders])

## §7 Plots for codes

In [ ]:
# W7.1  The object API: one Figure, several Axes, and you draw on each Axes
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
axes[0].imshow(H, cmap="Greys")
axes[0].set_title("Hamming [7,4]: H")
axes[0].set_xticks(range(7)); axes[0].set_yticks(range(3))
axes[1].imshow(HX, cmap="Greys", aspect="auto")      # HGP(R3, R3) from W3.2
axes[1].set_title("HGP(R3, R3): H_X")
plaquette = [hedge_index(3, 0, 0), vedge_index(3, 1, 0), hedge_index(3, 0, 1), vedge_index(3, 0, 0)]
plot_lattice(3, highlight=plaquette, labels=True, ax=axes[2], title="T_3 edge numbers; f(0,0) in green")
plt.tight_layout(); plt.show()

In [ ]:
# W7.2  Log-log plots: c * p^a is a straight line of slope a
ps = np.logspace(-4, -0.5, 80)
F_ham = [0, 0, 21, 35, 35, 21, 7, 1]                 # every error of weight >= 2 defeats the Hamming decoder
F_r5 = [0, 0, 0, 10, 5, 1]                           # R5 corrects up to 2 errors
P_unc = 1 - (1 - ps) ** 4                            # 4 bits sent without coding
P_ham = np.array([logical_rate_terms(F_ham, 7, p).sum() for p in ps])
P_r5 = np.array([logical_rate_terms(F_r5, 5, p).sum() for p in ps])

fig, ax = plt.subplots(figsize=(6.2, 4.6))
ax.loglog(ps, P_unc, label="4 bits, no code (slope 1)")
ax.loglog(ps, P_ham, label="Hamming [7,4], 4 bits (slope 2)")
ax.loglog(ps, P_r5, label="R5, 1 bit (slope 3)")
ax.set_xlabel("bit-flip probability p"); ax.set_ylabel("block failure probability")
ax.grid(which="both", alpha=0.25); ax.legend()
plt.show()
for p in (1e-1, 1e-2, 1e-3):                         # how fast does the leading term take over?
    exact = logical_rate_terms(F_ham, 7, p).sum()
    print(f"p = {p:<6g} Hamming block failure {exact:.3e}   leading term 21 p^2 = {21 * p**2:.3e}   "
          f"ratio {exact / (21 * p**2):.3f}")

## Exercises: write the functions in `drills.py`

Answer each set's "predict" questions (Exercises tab) on paper first. Then write the set and run its tests;
each cell below prints "not written yet" until you do.

| Set | Functions | Tests |
|---|---|---|
| P1 | `parity_check_from_columns` | `pytest detours/python -q -k p1` |
| P2 | `encode_all`, `weight_distribution` | `-k p2` |
| P3 | `in_rowspace`, `gf2_rref`, `gf2_kernel` (optional) | `-k p3` |
| P4 | `syndrome_histogram` | `-k p4` |
| P5 | `error_batches` (optional), `min_distance` | `-k p5` |
| P6 | `make_hamming_decoder`, `failure_counts` | `-k p6` |
| P7 | `loglog_slope` | `-k p7` |

In [ ]:
# X1  P1.1
try:
    H15 = dr.parity_check_from_columns(range(1, 16), 4)
    print(H15)
    print("same as W1.1 for columns 1..7:", np.array_equal(dr.parity_check_from_columns(range(1, 8), 3), H))
    print("Hamming [15, 11]: n =", H15.shape[1], " k = n - rank H =", 15 - gf2_rank(H15))
except NotImplementedError as err:
    print(err, "not written yet")

In [ ]:
# X2  P2.1, P2.2
try:
    C = dr.encode_all(G)
    print(C)
    print("all in ker H:", not (C.astype(np.int64) @ H.T % 2).any(), "  distinct:", len(np.unique(C, axis=0)))
    print("A_w, Hamming [7,4]         :", dr.weight_distribution(G))
    G8 = np.hstack([G, G.sum(axis=1, keepdims=True) % 2])    # append an overall parity bit
    print("A_w, extended Hamming [8,4]:", dr.weight_distribution(G8))
except NotImplementedError as err:
    print(err, "not written yet")

In [ ]:
# X3  P3.1, P3.2, P3.3 (optional)
try:
    print("in_rowspace(G, c):", dr.in_rowspace(G, c), "   in_rowspace(G, x):", dr.in_rowspace(G, x))
    R, piv = dr.gf2_rref(M)
    print("rref of M from W3.1:\n", R, "\npivot columns", piv)
except NotImplementedError as err:
    print(err, "not written yet")
try:
    K = dr.gf2_kernel(H)
    print("kernel basis of the Hamming H:\n", K)
    print("rank", gf2_rank(K), "  H K^T = 0:", not (H.astype(np.int64) @ K.T % 2).any())
except NotImplementedError as err:
    print(err, "not written yet")

In [ ]:
# X4  P4.1
try:
    for name, HH, w in [("Hamming [7,4]", H, 1), ("Hamming [7,4]", H, 2), ("R5", R5, 2)]:
        h = dr.syndrome_histogram(HH, w)
        print(f"{name:13s} weight <= {w}: {sum(h.values()):3d} errors, {len(h):2d} syndromes, "
              f"(errors per syndrome, how many syndromes): {sorted(Counter(h.values()).items())}")
except NotImplementedError as err:
    print(err, "not written yet")

In [ ]:
# X5  P5.1 (optional), P5.2
try:
    for chunk_sups, chunk_E in dr.error_batches(5, 2, 4):
        print(chunk_sups.tolist())
except NotImplementedError as err:
    print(err, "not written yet")
try:
    print("d(Hamming [7,4]) =", dr.min_distance(H, 4), "   d(R5) =", dr.min_distance(R5, 5))
    for d in (3, 4, 5):
        _, HZd = hypergraph_product(cycle_code(d), cycle_code(d))
        t0 = time.perf_counter()
        dist = dr.min_distance(HZd, 4)
        print(f"T_{d}: smallest nonzero x with H_Z x = 0 has weight {dist}   ({time.perf_counter() - t0:.2f} s)")
except NotImplementedError as err:
    print(err, "not written yet")

In [ ]:
# X6  P6.1, P6.2
try:
    print("Hamming [7,4], F_w for w = 0..3:", dr.failure_counts(dr.make_hamming_decoder(H), H, 3))
    print("R5 (W6.1),     F_w for w = 0..5:", dr.failure_counts(decode5, R5, 5))
except NotImplementedError as err:
    print(err, "not written yet")

In [ ]:
# X7  P7.1
try:
    small = ps < 1e-3
    for name, P in [("no code", P_unc), ("Hamming [7,4]", P_ham), ("R5", P_r5)]:
        print(f"{name:14s} slope for p in [1e-4, 1e-3]: {dr.loglog_slope(ps[small], P[small]):.3f}")
except NotImplementedError as err:
    print(err, "not written yet")

When every set is green, ask the detour chat for the Part I hand-off note. Part II opens with Lesson 2.